In [0]:
from datetime import datetime
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
road_schema = StructType([
    StructField("count_point_id", StringType()),   # cast to Int explicitly downstream
    StructField("year", StringType()),
    StructField("region_id", StringType()),
    StructField("region_name", StringType()),
    StructField("region_ons_code", StringType()),
    StructField("local_authority_id", StringType()),
    StructField("local_authority_name", StringType()),
    StructField("local_authority_code", StringType()),
    StructField("road_name", StringType()),
    StructField("road_category", StringType()),
    StructField("road_type", StringType()),
    StructField("start_junction_road_name", StringType()),
    StructField("end_junction_road_name", StringType()),
    StructField("easting", StringType()),
    StructField("northing", StringType()),
    StructField("latitude", StringType()),
    StructField("longitude", StringType()),
    StructField("link_length_km", StringType()),
    StructField("link_length_miles", StringType()),
    StructField("Extract_Time", StringType()),
    StructField("source", StringType()),
])
road_schema_format = ", ".join(
    f"{i.name} STRING" for i in road_schema.fields
)



In [0]:
traffic_schema = StructType([
    StructField("count_point_id", StringType()),
    StructField("year", StringType()),
    StructField("region_id", StringType()),
    StructField("region_name", StringType()),
    StructField("region_ons_code", StringType()),
    StructField("local_authority_id", StringType()),
    StructField("local_authority_name", StringType()),
    StructField("local_authority_code", StringType()),
    StructField("road_name", StringType()),
    StructField("road_category", StringType()),
    StructField("road_type", StringType()),
    StructField("start_junction_road_name", StringType()),
    StructField("end_junction_road_name", StringType()),
    StructField("easting", StringType()),
    StructField("northing", StringType()),
    StructField("latitude", StringType()),
    StructField("longitude", StringType()),
    StructField("estimation_method", StringType()),
    StructField("estimation_method_detailed", StringType()),
    StructField("direction_of_travel", StringType()),
    StructField("pedal_cycles", StringType()),
    StructField("two_wheeled_motor_vehicles", StringType()),
    StructField("cars_and_taxis", StringType()),
    StructField("buses_and_coaches", StringType()),
    StructField("LGVs", StringType()),
    StructField("HGVs_2_rigid_axle", StringType()),
    StructField("HGVs_3_rigid_axle", StringType()),
    StructField("HGVs_4_or_more_rigid_axle", StringType()),
    StructField("HGVs_3_or_4_articulated_axle", StringType()),
    StructField("HGVs_5_articulated_axle", StringType()),
    StructField("HGVs_6_articulated_axle", StringType()),
    StructField("all_HGVs", StringType()),
    StructField("all_motor_vehicles", StringType()),
    StructField("source", StringType()),
    StructField("link_length_km", StringType()),
    StructField("link_length_miles", StringType()),
])
traffic_schema_format = ", ".join(
    f"{i.name} STRING" for i in traffic_schema.fields
)


In [0]:
file_path = "/Volumes/dev_catalog/landing/landing_vol/raw_roads/"
file_path2 = "/Volumes/dev_catalog/landing/landing_vol/raw_traffic/"
checkpoint_location_raw_roads = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_roads"
checkpoint_location_raw_roads_schema = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_roads_schema"

checkpoint_location_raw_traffic = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_traffic"
checkpoint_location_raw_traffic_schema = "/Volumes/dev_catalog/checkpoints/checkpoint_vol/raw_traffic_schema"
notebook_name = "03_load_to_bronze_incremental_batch"

In [0]:
def log_pipeline_error(step_name, error):
    spark.createDataFrame(
        [(notebook_name, step_name, str(error), datetime.now())],
        ["notebook", "step", "error_message", "error_time"]
    ).write.mode("append").saveAsTable("dev_catalog.default.pipeline_errors")

In [0]:
try:
    df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.schemaLocation", checkpoint_location_raw_roads_schema) \
        .option("cloudFiles.allowOverwrites", "true") \
        .option("cloudFiles.schemaHints", road_schema_format)\
        .option("cloudFiles.schemaEvolutionMode", "rescue")\
        .load(file_path)
    df = df.withColumn("Extract_Time", current_timestamp())
    df = df.withColumn("source" , ifnull(col("source"),lit("A")))
  
except Exception as e:
    log_pipeline_error("read_csv_road", e)
    raise


In [0]:
try:
    df.writeStream.option("checkpointLocation", checkpoint_location_raw_roads) \
      .trigger(availableNow=True) \
      .table("dev_catalog.bronze.raw_roads")
  
except Exception as e:
    log_pipeline_error("read_csv_road", e)
    raise

in pyspark api way
try:
    df_traffic = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.schemaLocation", checkpoint_location_raw_traffic_schema) \
        .option("cloudFiles.schemaEvolutionMode", "rescue")\
        .load(file_path2)
    df_traffic = df_traffic.withColumn("Extract_Time", current_timestamp())
  
except Exception as e:
    log_pipeline_error("read_csv_traffic", e)
    raise


In [0]:
try:
    df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.schemaLocation", checkpoint_location_raw_traffic_schema) \
        .option("cloudFiles.allowOverwrites", "true") \
        .option("cloudFiles.schemaHints", traffic_schema_format)\
        .option("cloudFiles.schemaEvolutionMode", "rescue")\
        .load(file_path)
    df = df.withColumn("Extract_Time", current_timestamp())
    df = df.withColumn("source" , ifnull(col("source"),lit("A")))
  
except Exception as e:
    log_pipeline_error("read_csv_traffic", e)
    raise


In [0]:
try:
    df.writeStream.option("checkpointLocation", checkpoint_location_raw_traffic) \
      .trigger(availableNow=True) \
      .table("dev_catalog.bronze.raw_traffic")
  
except Exception as e:
    log_pipeline_error("write_csv_traffic", e)
    raise

COPY INTO dev_catalog.bronze.raw_traffic
FROM (Select * ,
coalesce(ifnull(source) , 'A' ) as source,
current_timestamp() as Extract_Time
from "/Volumes/dev_catalog/landing/landing_vol/raw_traffic/")
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true' , 'rescuedDataColumn' = '_rescued_data', 'schemaHints' = 'source STRING')
COPY_OPTIONS ('mergeSchema' = 'true')
